# Pre-Requisites

## Load 'users_001.csv'

In [0]:
df = spark.read.csv(
    path="/Volumes/merit_catalog/quickstart_schema/sandbox/dataset/user_dataset/users_001.csv",
    header=True,
    inferSchema=True,
)
df.limit(4).display()

## Writing it as Delta Table
- When it is loaded into the dataframe it is written as Delta Table

In [0]:
df.write.mode("overwrite").saveAsTable("merit_catalog.quickstart_schema.users_new")

## Read csv

In [0]:
spark.read.table("merit_catalog.quickstart_schema.users_new").display()

# Schema Enforcement
- When a csv file with different schema is given the update is not allowed

In [0]:
spark.read.csv(
    path="/Volumes/merit_catalog/quickstart_schema/sandbox/dataset/user_dataset/users_006_new_column_education.csv",
    header=True,
    inferSchema=True,
).write.mode("append").saveAsTable("merit_catalog.quickstart_schema.users_new")

# Schema Evolution (Extra column)
- When a csv file with different schema (Extra col) is given the update is allowed
- Option is added to enable Schema Evolution

In [0]:
spark.read.csv(
    path="/Volumes/merit_catalog/quickstart_schema/sandbox/dataset/user_dataset/users_006_new_column_education.csv",
    header=True,
    inferSchema=True,
).write.option("mergeSchema",True).mode("append").saveAsTable("merit_catalog.quickstart_schema.users_new")

## Read the file after update
1. Old data id 1-5 (By default - NULL)
2. New data id 2501-2505 (Values for Degree)

In [0]:
from pyspark.sql.functions import col

spark.read.table("merit_catalog.quickstart_schema.users_new").filter(col("id").between(1,5)).display()

In [0]:
from pyspark.sql.functions import col

spark.read.table("merit_catalog.quickstart_schema.users_new").filter(col("id").between(2501,2505)).display()

# Schema Evolution (Less columns)
- When a csv file with different schema (Less cols) is given the update is allowed

In [0]:
spark.read.csv(
    path="/Volumes/merit_catalog/quickstart_schema/sandbox/dataset/user_dataset/users_012_less_columns.csv",
    header=True,
    inferSchema=True,
).write.mode("append").saveAsTable("merit_catalog.quickstart_schema.users_new")

## Read the file after update
- New data added will have other columns as null and added columns have value

In [0]:
spark.read.table("merit_catalog.quickstart_schema.users_new").display()

# Schema Evolution (Data Type mismatch)
- When a csv file with different schema (Col with data type mismatch) is given the update is not allowed

In [0]:
spark.read.csv(
    path="/Volumes/merit_catalog/quickstart_schema/sandbox/dataset/user_dataset/users_013_datatype_mismatch_dob.csv",
    header=True,
    inferSchema=True,
).write.mode("append").saveAsTable("merit_catalog.quickstart_schema.users_new")

## Append after giving merge schema = True

In [0]:
spark.read.csv(
    path="/Volumes/merit_catalog/quickstart_schema/sandbox/dataset/user_dataset/users_013_datatype_mismatch_dob.csv",
    header=True,
    inferSchema=True,
).write.option("mergeSchema",True).mode("append").saveAsTable("merit_catalog.quickstart_schema.users_new")

# Schema Evolution (Column name mismatch)
- When a csv file with different schema (Col with name mismatch) is given the update is not allowed
- Will not allow to add column with different clumn name (Now no error as executed the MergeSchema first)
- MergeSchema will allow you to add But not recognised as that id,name column added as new col.

In [0]:
spark.read.csv(
    path="/Volumes/merit_catalog/quickstart_schema/sandbox/dataset/user_dataset/users_013_less_columns_with_different_column_names.csv",
    header=True,
    inferSchema=True,
).write.mode("append").saveAsTable("merit_catalog.quickstart_schema.users_new")

In [0]:
spark.read.table("merit_catalog.quickstart_schema.users_new").display()

In [0]:
spark.read.csv(
    path="/Volumes/merit_catalog/quickstart_schema/sandbox/dataset/user_dataset/users_013_less_columns_with_different_column_names.csv",
    header=True,
    inferSchema=True,
).write.option("mergeSchema",True).mode("append").saveAsTable("merit_catalog.quickstart_schema.users_new")

# Schema Evolution (Column name mismatch - Alias name)
- When a csv file with different schema (Col with name mismatch) is given the update is not allowed
- MergeSchema will allow you to add But not recognised as that id,name column added as new col.
- Now to avoid point 2, we give alias name to add it to the same column.

In [0]:
spark.read.csv(
    path="/Volumes/merit_catalog/quickstart_schema/sandbox/dataset/user_dataset/users_014_less_columns_with_different_column_names.csv",
    header=True,
    inferSchema=True,
).select(col("u_id").alias("id"), col("f_name").alias("name")).write.mode("append").saveAsTable("merit_catalog.quickstart_schema.users_new")

In [0]:
spark.read.table("merit_catalog.quickstart_schema.users_new").display()